# Sheet 6 — Seventh chords, edge Tonnetze and non-orientable surfaces (Chapter 7)

We (a) reproduce the slide "V − E + F = 1" from the hexagonal diagram, (b) test orientability,
(c) build the actual pitch-class complex of whole-tone shell chords, and (d) explore the space of edge-labelled gluings.

In [1]:
# Setup: make the repository importable whether or not `pip install -e .` was run
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import numpy as np, networkx as nx, matplotlib.pyplot as plt
from tonnetzkit import *
%matplotlib inline
import collections

### (a) The hexagon with antipodal boundary identification

In [2]:
pts = [(q, r) for q in range(-2, 3) for r in range(-2, 3) if max(abs(q), abs(r), abs(q+r)) <= 2]
S = set(pts); tris = set()
for (q, r) in pts:
    for tri in (((q,r),(q+1,r),(q,r+1)), ((q+1,r),(q+1,r-1),(q,r))):
        if all(p in S for p in tri): tris.add(tuple(sorted(tri)))
on_bdry = lambda p: max(abs(p[0]), abs(p[1]), abs(p[0]+p[1])) == 2
label = lambda p: min(p, (-p[0], -p[1])) if on_bdry(p) else p
K = SimplicialComplex([tuple(label(p) for p in t) for t in tris])
print(K.summary()); print('=>', classify_surface(K))

{'V': 13, 'E': 36, 'F': 24, 'chi': 1, 'betti_Q': (1, 0, 0), 'betti_Z2': (1, 1, 1), 'max_edge_degree': 2, 'orientable': False}
=> real projective plane


Betti numbers differ between $\mathbb{Q}$ and $\mathbb{Z}_2$ — the fingerprint of the projective plane ($H_1 = \mathbb{Z}_2$).

### (b) The honest pitch-class complex of shell chords

In [3]:
wt = [0, 2, 4, 6, 8, 10]
dom = [essential_notes(r, 'dom7') for r in wt]; m9 = [essential_notes(r, 'maj9') for r in wt]
for name, ch in [('dom7 shells', dom), ('maj9 shells', m9), ('both', dom + m9)]:
    Kc = chord_complex(ch); print(f'{name:12s}', Kc.summary(), '->', classify_surface(Kc))

dom7 shells  {'V': 6, 'E': 15, 'F': 6, 'chi': -3, 'betti_Q': (1, 4, 0), 'betti_Z2': (1, 4, 0), 'max_edge_degree': 2, 'orientable': True} -> surface with boundary, chi=-3, orientable
maj9 shells  {'V': 6, 'E': 12, 'F': 6, 'chi': 0, 'betti_Q': (1, 1, 0), 'betti_Z2': (1, 1, 0), 'max_edge_degree': 2, 'orientable': True} -> annulus (cylinder)
both         {'V': 6, 'E': 15, 'F': 12, 'chi': 3, 'betti_Q': (1, 0, 2), 'betti_Z2': (1, 0, 2), 'max_edge_degree': 3, 'orientable': None} -> not a surface (some edge in >2 triangles)


### (c) Edge-labelled gluings: a census

In [4]:
odd = [1, 3, 5, 7, 9, 11]
faces = [tuple(sorted(essential_notes(r, 'dom7'))) for r in odd] + [tuple(sorted(essential_notes(r, 'maj9'))) for r in odd]
census = sample_edge_tonnetze(faces, n=20000, seed=1)
tot = sum(census.values())
for k, v in sorted(census.items(), key=lambda kv: -kv[1]): print(f'{k:32s} {100*v/tot:6.2f}%')

non-orientable, 6 cross-caps      37.61%
non-orientable, 5 cross-caps      25.83%
non-orientable, 7 cross-caps      23.40%
non-orientable, 4 cross-caps       9.33%
non-orientable, 3 cross-caps       2.23%
Klein bottle                       0.43%
orientable genus 2                 0.38%
disconnected                       0.34%
orientable genus 3                 0.30%
torus                              0.07%
projective plane                   0.06%
sphere                             0.01%


## Exercises
1. Find (by search) an edge gluing of the 12 shell chords that is a **sphere**. How many vertices does it have?
2. Use `essential_notes(r, 'maj9_jazz')` (3rd, 7th, 9th) instead. Does the census change?
3. Prove that a closed surface with $\chi = 1$ cannot be orientable.
<details><summary><b>Show solution</b></summary>

```python
import random
rng = random.Random(0)
while True:
    g = random_edge_gluing(faces, rng)
    if g.surface_name() == 'sphere': print(g.V, g.E, g.F); break   # V = 8, E = 18, F = 12
```
2. The jazz shells {3,7,9} = {r+4, r+11, r+2} lie in *both* whole-tone scales, so the faces must be rebuilt over all 12 roots; rerun and compare.
3. Orientable closed surfaces have χ = 2 − 2g, which is even; 1 is odd.

</details>